In [18]:
from cybergear_packet import CybergearPacket
import socket
import struct

def send_frame_direct(interface_name, can_id, data):
    """
    Send a CAN frame directly to a CAN interface without using ROS bridges
    
    Args:
        packet: CybergearPacket instance
        interface_name (str): CAN interface name (e.g., 'can0')
        can_id (int): CAN ID for the message
        data (list): Data bytes to send (up to 8 bytes)
    """
    
    try:
        # Create and bind socket
        s = socket.socket(socket.AF_CAN, socket.SOCK_RAW, socket.CAN_RAW)
        s.bind((interface_name,))
        
        # Prepare data - ensure it's exactly 8 bytes
        can_dlc = min(len(data), 8)
        data_padded = data[:8] + [0] * (8 - len(data[:8]))
        
        # CAN frame format: <CAN_ID><DLC><DATA[8]>
        # Use proper struct packing for CAN frame
        can_frame_format = "=IB3x8B"  # ID(4) + DLC(1) + padding(3) + data(8)
        
        can_frame = struct.pack(can_frame_format, 
                               can_id,           # CAN ID
                               can_dlc,          # Data length
                               *data_padded)     # 8 bytes of data
        
        # Send the frame
        bytes_sent = s.send(can_frame)
        print(f"Frame sent to {interface_name}: ID={hex(can_id)}, DLC={can_dlc}, data={data[:can_dlc]}")
        print(f"Bytes sent: {bytes_sent}")
        
    except socket.error as e:
        print(f"Socket error: {e}")
        # Check if CAN interface exists
        import subprocess
        try:
            result = subprocess.run(['ip', 'link', 'show', interface_name], 
                                  capture_output=True, text=True)
            if result.returncode != 0:
                print(f"CAN interface '{interface_name}' may not exist or be up")
                print("Try: sudo ip link set can0 up type can bitrate 1000000")
        except:
            pass
    except Exception as e:
        print(f"Unexpected error: {e}")
    finally:
        if 's' in locals():
            s.close()

In [27]:
packet = CybergearPacket(0, 127)

send_frame_direct('can1', 127, [0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00])

Frame sent to can1: ID=0x7f, DLC=8, data=[0, 0, 0, 0, 0, 0, 0, 0]
Bytes sent: 16


In [ ]:
import threading

def receive_can_messages(interface_name, callback):
    """
    Listen for CAN messages on the given interface and call the callback with (can_id, data) when a frame is received.
    """
    def listener():
        try:
            s = socket.socket(socket.AF_CAN, socket.SOCK_RAW, socket.CAN_RAW)
            s.bind((interface_name,))
            while True:
                frame = s.recv(16)
                can_id, can_dlc = struct.unpack("=IB3x", frame[:8])
                data = list(struct.unpack("8B", frame[8:16]))[:can_dlc]
                callback(can_id, data)
        except Exception as e:
            print(f"Error in CAN listener: {e}")
        finally:
            if 's' in locals():
                s.close()
    
    thread = threading.Thread(target=listener, daemon=True)
    thread.start()